# Studio 3.3 — The Diet Problem: Feed a Football Team at McDonald's

**OPIM 5641 · Business Decision Modeling · Dr. Dave Wanik, UConn**

The team bus pulls into McDonald's. **53 players**, one day of meals, and the athletic department is
paying. What's the cheapest order that keeps everyone fueled?

It's Stigler's diet again, with **three changes**:

| | Stigler | McDonald's |
|---|---|---|
| **Decision variable** | dollars per day on each food | **how many of each menu item** to order for the whole team |
| **Objective** | add up the dollars | add up **price × how many** |
| **Rules** | every nutrient has a minimum (`>=`) | some are minimums (`>=`), some are **maximums** (`<=`), and they are **per player**, so multiply by the team size |

Plus one new rule for variety: **nobody gets more than 4 of the same item.**

**How this notebook works.** All of the code is already here. You do not have to write a model
from scratch. Your job is to **run it, read it, and change it**: each experiment has a clearly marked
cell where you plug in a scenario, re-run, and see what happens.

**About the data (it is real):**
- **Nutrition** comes from **MenuStat**, the New York City Health Department's database of chain
  restaurant nutrition (2018 edition).
- **Prices** are McDonald's menu prices as listed on fastfoodmenuprices.com (updated July 11, 2026).
  Prices vary by location, so yours may be a little different.
- The menu has the items that appear in both sources.

**Before you solve it, guess:** what's the cheapest bill per player? Write it down.

## Import modules (download Pyomo)

Run this first, every time. It installs Pyomo and the `cbc` solver in Colab.

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES
import pandas as pd

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solver if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

## 1 · The data

Two CSV files: the **menu** (one row per item) and the **rules** (per player, per day).

`menu_file` is the only thing that says "McDonald's" in this whole notebook. Remember that - your
assignment will be to swap in another restaurant.

In [ ]:
# read the data
data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio3/menus/'
menu_file = 'mcdonalds_menu.csv'
rules_file = 'team_rules.csv'

menu_df = pd.read_csv(data_folder + menu_file)
rules_df = pd.read_csv(data_folder + rules_file)

menu_df

In [ ]:
# the rules, per player per day - notice calories has BOTH a min and a max
rules_df

In [ ]:
# ---------- 1. THE DATA ----------
# the menu items - every column after 'price' is a nutrient
items = list(menu_df['item'])
nutrient_columns = list(menu_df.columns[3:])

# price[item] and nutrition[item][nutrient] (for ONE of that item)
price = {}
nutrition = {}
for i in range(len(menu_df)):
    item = menu_df.loc[i, 'item']
    price[item] = float(menu_df.loc[i, 'price'])
    nutrition[item] = {}
    for nutrient in nutrient_columns:
        nutrition[item][nutrient] = float(menu_df.loc[i, nutrient])

# a readable label for each nutrient (the rules file has nicer ones)
label = {}
for nutrient in nutrient_columns:
    label[nutrient] = nutrient

# the rules per player: one dictionary of minimums, one of maximums
player_min = {}
player_max = {}
for i in range(len(rules_df)):
    nutrient = rules_df.loc[i, 'nutrient']
    label[nutrient] = rules_df.loc[i, 'label']
    if rules_df.loc[i, 'rule'] == 'min':
        player_min[nutrient] = int(rules_df.loc[i, 'per_player'])
    else:
        player_max[nutrient] = int(rules_df.loc[i, 'per_player'])

print("minimums per player:", player_min)
print("maximums per player:", player_max)
print()
print("Big Mac: $", price['Big Mac'], nutrition['Big Mac'])

**The scoreboard.** An empty dictionary. Every time you run the SOLVE AND REPORT cell, it adds one
entry with `scoreboard[scenario] = ...` (a new key adds a new entry; there is no `append` for
dictionaries). Run this cell **once** - running it again wipes the scoreboard clean.

In [ ]:
# an empty scoreboard: scenario name -> cost
scoreboard = {}

## 2 · The scenario

This is the cell you play with. Leave it alone for the first run.

In [ ]:
# ---------- 2. THE SCENARIO (change things here) ----------
scenario = "1 base"
team_size = 53              # players on the roster
max_of_one_item = 4         # nobody gets more than 4 of the same item
low = dict(player_min)      # a fresh copy of the minimums per player
high = dict(player_max)     # a fresh copy of the maximums per player
banned = []                 # items nobody will eat
must_have = {}              # items everybody gets, and how many each

# Scenario 2: no variety rule (remove the # from the next two lines)
# scenario = "2 no variety rule"
# max_of_one_item = 100

# Scenario 3: coach's order - everybody gets a Big Mac and a medium fries
# scenario = "3 coach: Big Mac + fries"
# must_have = {'Big Mac' : 1, 'Medium French Fries' : 1}

# Scenario 4: the team doctor cuts the sodium limit to 5,000 mg
# scenario = "4 sodium 5,000"
# high['sodium'] = 5000

# Scenario 5: break it - the FDA says 2,300 mg of sodium a day
# scenario = "5 sodium 2,300"
# high['sodium'] = 2300

# Scenario 6: your own idea. Some things you can change:
#   team_size = 11                      a soccer team instead
#   low['protein'] = 200                linemen need more protein
#   high['sugar'] = 50                  the nutritionist hates sugar
#   banned = ['Filet-O-Fish']           nobody eats the fish
#   must_have = {'Apple Slices' : 2}    everybody gets two bags of apple slices


## 3 · The model

Same shape as Stigler. Read it once, then leave it alone: every scenario above works without
changing a line here.

In [ ]:
# ---------- 3. THE MODEL (you only change the last part, if you want to) ----------
model = ConcreteModel()

# how many of each item to order for the whole team
model.x = Var(items, domain=NonNegativeReals)

# objective: the total bill
obj_expr = 0
for item in items:
    obj_expr += price[item] * model.x[item]

model.cost = Objective(
                      expr = obj_expr,
                      sense = minimize)

model.constraints = ConstraintList()

# the minimums (multiply by the team size!)
for nutrient in low:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr >= low[nutrient] * team_size)

# the maximums
for nutrient in high:
    nutrient_expr = 0
    for item in items:
        nutrient_expr += nutrition[item][nutrient] * model.x[item]
    model.constraints.add(nutrient_expr <= high[nutrient] * team_size)

# variety: no more than max_of_one_item of any item per player
for item in items:
    model.constraints.add(model.x[item] <= max_of_one_item * team_size)

# scenario rules: banned items, and items everybody gets
for item in banned:
    model.constraints.add(model.x[item] == 0)
for item in must_have:
    model.constraints.add(model.x[item] >= must_have[item] * team_size)

# ---------- YOUR OWN CONSTRAINT (optional) ----------
# You can also write a constraint yourself. For example, at least as many fries as cheeseburgers:
# model.constraints.add(model.x['Medium French Fries'] >= model.x['Cheeseburger'])


print("Number of variables =", len(model.x))
print("Number of constraints =", len(model.constraints))

## 4 · Solve it and see the order

This cell solves the model, prints the order, checks the nutrition per player, draws the order, and
writes the result on the scoreboard.

In [ ]:
# ---------- 4. SOLVE AND REPORT (you never change this cell) ----------
results = SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)

print("SCENARIO:", scenario)
if results.solver.termination_condition == TerminationCondition.optimal:
    per_player = round(model.cost() / team_size, 2)
    print("Total bill = $", round(model.cost(), 2))
    print("Per player = $", per_player)

    print()
    print("The order:")
    names = []
    counts = []
    for item in items:
        if model.x[item]() > 0.000001:
            print(item, ":", round(model.x[item](), 1), "total (", round(model.x[item]() / team_size, 2), "per player )")
            names.append(item)
            counts.append(model.x[item]() / team_size)

    print()
    print("Per player per day:")
    for nutrient in nutrient_columns:
        total = 0
        for item in items:
            total += nutrition[item][nutrient] * model.x[item]()
        rule_text = ""
        if nutrient in low:
            rule_text += " min " + str(low[nutrient])
        if nutrient in high:
            rule_text += " max " + str(high[nutrient])
        if rule_text == "":
            rule_text = " no rule"
        print(label[nutrient], ":", round(total / team_size, 1), "(" + rule_text + " )")

    # add this scenario to the scoreboard dictionary: scoreboard[key] = value
    scoreboard[scenario] = per_player

    # a picture of the order
    barh(names, counts, color='darkorange')
    xlabel('how many per player')
    title(scenario + ':  $%.2f per player' % per_player)
    tight_layout()
    savefig('fig_team_order.png')
    show()
else:
    print("INFEASIBLE! No order can satisfy every rule at the same time.")
    print("Loosen a rule and try again.")

How close was your guess?

Look at the per-player numbers. Which rules are **binding** (sitting exactly on the min or the max)?
Those are what's driving the bill.

### Try it: play with the scenarios

For each scenario: remove the `#` from that scenario's lines in **THE SCENARIO** cell, then run the
scenario cell, the model cell, and the solve cell. Put the `#` back before you try the next one.

| Scenario | What changes | Cost per player | What changed in the order? |
|---|---|---|---|
| 1 base | nothing | | |
| 2 | no variety rule | | |
| 3 | coach: everybody gets a Big Mac and fries | | |
| 4 | sodium limit down to 5,000 mg | | |
| 5 | sodium limit down to 2,300 mg (the FDA's number) | | |
| 6 | your own idea | | |

Then try writing **one constraint of your own** at the bottom of the model cell. The example there
says "at least as many fries as cheeseburgers". What rule would your team want?

In [ ]:
# the scoreboard so far
print(scoreboard)

## Your deliverable: the team order card

When you've played your scenarios, run the cell below to draw your scoreboard, then answer the three
questions underneath it. That's what you turn in tonight.

In [ ]:
# draw the scoreboard
scenario_names = []
dollars = []
for name in scoreboard:
    scenario_names.append(name)
    dollars.append(scoreboard[name])

barh(scenario_names, dollars, color='steelblue')
xlabel('$ per player')
title('Feed the team: what did each rule cost?')
gca().invert_yaxis()
tight_layout()
savefig('fig_scoreboard.png')
show()

**Double-click this cell and answer:**

1. **Scenario 2:** what did the cheapest order look like with no variety rule, and how much did
   variety cost per player?

   *your answer*

2. **Scenarios 4 and 5:** what did cutting the sodium to 5,000 mg cost per player? What happened at
   2,300 mg, and what does that tell you about a day of McDonald's?

   *your answer*

3. **Scenario 6:** what did you change, and what did it cost the team? Was it worth it?

   *your answer*

**One more thing to think about:** you can't order 1.58 McChickens per player. What would you change
in the model to get whole numbers? (That's integer programming - coming in Module 5.)

## Bottom line

- Stigler's diet and the McDonald's order are **the same model with different CSV files.**
- A minimum is a `>=` constraint, a maximum is a `<=` constraint. Same loop, different sign.
- **Infeasible** means the rules contradict each other. The fix is to loosen a rule, not to fix the code.
- Swap `menu_file` and the same notebook works for any restaurant. That's your assignment.

**Before you move on:** File → Save a copy in GitHub.